In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")  # base de datos temporal, solo en memoria
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE job_postings (
        id INTEGER PRIMARY KEY,
            company TEXT,
                role TEXT,
                    salary INTEGER,
                        posted_date TEXT
)
""")

sample_data = [
        (1, "Equifax", "Data Engineer", 85000, "2026-01-05"),
            (2, "Equifax", "Data Engineer", 92000, "2026-01-12"),
                (3, "Equifax", "AI Engineer", 98000, "2026-01-08"),
                    (4, "Google", "Data Engineer", 130000, "2026-01-03"),
                        (5, "Google", "Data Engineer", 125000, "2026-01-20"),
                            (6, "Google", "AI Engineer", 145000, "2026-01-15"),
                                (7, "Amazon", "Data Engineer", 110000, "2026-01-07"),
                                    (8, "Amazon", "AI Engineer", 120000, "2026-01-18"),
                                        (9, "Microsoft", "Data Engineer", 115000, "2026-01-10"),
                                            (10, "Microsoft", "AI Engineer", 128000, "2026-01-22"),
                                                (11, "Microsoft", "AI Data Engineer", 128000, "2026-01-28"),
]

cursor.executemany("INSERT INTO job_postings VALUES (?, ?, ?, ?, ?)", sample_data)
conn.commit()

pd.read_sql("SELECT * FROM job_postings", conn)


,id,company,role,salary,posted_date
0,1,Equifax,Data Engineer,85000,2026-01-05
1,2,Equifax,Data Engineer,92000,2026-01-12
2,3,Equifax,AI Engineer,98000,2026-01-08
3,4,Google,Data Engineer,130000,2026-01-03
4,5,Google,Data Engineer,125000,2026-01-20
5,6,Google,AI Engineer,145000,2026-01-15
6,7,Amazon,Data Engineer,110000,2026-01-07
7,8,Amazon,AI Engineer,120000,2026-01-18
8,9,Microsoft,Data Engineer,115000,2026-01-10
9,10,Microsoft,AI Engineer,128000,2026-01-22


Tu ejercicio: usando la tabla job_postings, escribe una consulta que asigne un número de fila a cada oferta, reiniciando el conteo por cada company, ordenando de mayor a menor salary — para que el número 1 sea siempre la oferta mejor pagada de esa empresa.

ROW_NUMBER() es una función de ventana — a diferencia de GROUP BY (que colapsa filas en resúmenes), esta función te deja ver cada fila individual, pero le agrega un número calculado. Es como enumerar personas dentro de cada fila de un cine, sin fusionar las filas.

Su sintaxis:


ROW_NUMBER() OVER (PARTITION BY columna_agrupacion ORDER BY columna_orden)

PARTITION BY → dice "reinicia el conteo cada vez que cambie esta columna". Es el equivalente a lo que haría GROUP BY, pero sin fusionar filas — cada grupo (partición) cuenta desde 1 de nuevo.
ORDER BY → define el criterio para decidir quién es el número 1, 2, 3... dentro de cada grupo. Si quieres que el más alto sea el 1, necesitas orden descendente (DESC); si no especificas nada, por defecto es ascendente.


Estructura de ayuda:

pd.read_sql("""
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY ___ ORDER BY ___ ___) AS rn
    FROM job_postings
""", conn)


In [6]:
pd.read_sql("""
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY company ORDER BY salary DESC) AS rn
    FROM job_postings
""", conn)

,id,company,role,salary,posted_date,rn
0,8,Amazon,AI Engineer,120000,2026-01-18,1
1,7,Amazon,Data Engineer,110000,2026-01-07,2
2,3,Equifax,AI Engineer,98000,2026-01-08,1
3,2,Equifax,Data Engineer,92000,2026-01-12,2
4,1,Equifax,Data Engineer,85000,2026-01-05,3
5,6,Google,AI Engineer,145000,2026-01-15,1
6,4,Google,Data Engineer,130000,2026-01-03,2
7,5,Google,Data Engineer,125000,2026-01-20,3
8,10,Microsoft,AI Engineer,128000,2026-01-22,1
9,11,Microsoft,AI Data Engineer,128000,2026-01-28,2


Entonces lo que es importante es entender que se va a asignar una nueva columna a los valores que trae el query ademas una de las partes mas importantes es el PARTITION BY el cual me dice como es que se van partir los datos, osea porque variable en este caso es por company.

Quitando el DESC de la consulta, ORDER BY salary se comporta como ORDER BY salary ASC — y el rn=1 de cada empresa pasaría a ser el salario más bajo, no el más alto. Justo lo contrario de lo que pedía el ejercicio original.

In [7]:
pd.read_sql("""
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY company ORDER BY salary ) AS rn
    FROM job_postings
""", conn)

,id,company,role,salary,posted_date,rn
0,7,Amazon,Data Engineer,110000,2026-01-07,1
1,8,Amazon,AI Engineer,120000,2026-01-18,2
2,1,Equifax,Data Engineer,85000,2026-01-05,1
3,2,Equifax,Data Engineer,92000,2026-01-12,2
4,3,Equifax,AI Engineer,98000,2026-01-08,3
5,5,Google,Data Engineer,125000,2026-01-20,1
6,4,Google,Data Engineer,130000,2026-01-03,2
7,6,Google,AI Engineer,145000,2026-01-15,3
8,9,Microsoft,Data Engineer,115000,2026-01-10,1
9,10,Microsoft,AI Engineer,128000,2026-01-22,2


Siguiente concepto: RANK() vs DENSE_RANK() vs ROW_NUMBER()

Ya conoces ROW_NUMBER(): siempre da números consecutivos únicos (1, 2, 3, 4...), sin importar si hay empates en el valor que ordenas.

RANK() y DENSE_RANK() sí le prestan atención a los empates (dos filas con el mismo valor). Antes de explicarte la diferencia entre ellos, quiero que la descubras tú:

Ejercicio de exploración: Modifica tus datos de prueba para que dos ofertas de la misma empresa tengan exactamente el mismo salario (por ejemplo, dos ofertas de Equifax en $92,000). Luego corre las tres funciones lado a lado sobre esos datos:


In [8]:
pd.read_sql("""
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY company ORDER BY salary DESC) AS rn,
        RANK() OVER (PARTITION BY company ORDER BY salary DESC) AS rank_val,
        DENSE_RANK() OVER (PARTITION BY company ORDER BY salary DESC) AS dense_rank_val
    FROM job_postings
""", conn)

,id,company,role,salary,posted_date,rn,rank_val,dense_rank_val
0,8,Amazon,AI Engineer,120000,2026-01-18,1,1,1
1,7,Amazon,Data Engineer,110000,2026-01-07,2,2,2
2,3,Equifax,AI Engineer,98000,2026-01-08,1,1,1
3,2,Equifax,Data Engineer,92000,2026-01-12,2,2,2
4,1,Equifax,Data Engineer,85000,2026-01-05,3,3,3
5,6,Google,AI Engineer,145000,2026-01-15,1,1,1
6,4,Google,Data Engineer,130000,2026-01-03,2,2,2
7,5,Google,Data Engineer,125000,2026-01-20,3,3,3
8,10,Microsoft,AI Engineer,128000,2026-01-22,1,1,1
9,11,Microsoft,AI Data Engineer,128000,2026-01-28,2,1,1


RANK() — cuando hay empate, ambas filas comparten el mismo puesto (1 y 1). Pero para la siguiente fila, cuenta cuántas filas "gastó" el empate y salta esos puestos: como dos filas ocuparon el puesto 1, la siguiente no es 2, es 3 — como si dijera "dos personas empataron en primer lugar, entonces no hay segundo lugar, el siguiente es tercero".

DENSE_RANK() — también da el mismo puesto a los empatados (1 y 1), pero a la siguiente fila la numera de forma consecutiva sin huecos: simplemente "el siguiente puesto distinto" es 2, sin importar cuántas filas empataron antes. "Dense" (denso) se refiere justo a eso: no deja huecos en la secuencia de números.

La regla en una frase: RANK deja huecos después de un empate (refleja cuántas posiciones "ocupó" el empate); DENSE_RANK nunca deja huecos, solo cuenta valores distintos.

------------------------------------------------------------------------------

Siguiente tema: funciones de agregación como funciones de ventana

Ya conoces SUM(), AVG(), COUNT(), MAX(), MIN() en su forma clásica con GROUP BY (colapsando filas). La buena noticia es que las mismas funciones que ya conoces también funcionan como funciones de ventana — solo cambias GROUP BY por OVER (PARTITION BY ...), y automáticamente dejan de colapsar filas.

Ejemplo de fundamento:

sql

SELECT *,
    AVG(salary) OVER (PARTITION BY company) AS avg_salary_company
FROM job_postings

Esto no colapsa nada — sigues viendo las 10 (u 11, con tu empate) filas originales, pero cada una ahora "sabe" cuál es el salario promedio de su propia empresa. Es útil, por ejemplo, para comparar cada oferta individual contra el promedio de su grupo, sin perder el detalle de la oferta.

Nota importante: aquí no llevas ORDER BY dentro del OVER() — porque no estás ordenando ni enumerando nada, solo estás agregando un cálculo (promedio, suma, etc.) que aplica igual a todas las filas de esa partición.

----------------------------------------------------------------------------

Tu ejercicio: escribe una consulta que, para cada oferta de trabajo, muestre:

Todos los datos originales
El salario promedio de su empresa (avg_salary_company)
Cuánto se aleja esa oferta específica del promedio de su empresa (salary - avg_salary_company)

In [9]:
pd.read_sql("""
    SELECT *,
        AVG(salary) OVER (PARTITION BY company) AS avg_salary_company, salary - AVG(salary) OVER (PARTITION BY company) AS salary_diff_avg
        
    FROM job_postings
""", conn)

,id,company,role,salary,posted_date,avg_salary_company,salary_diff_avg
0,7,Amazon,Data Engineer,110000,2026-01-07,115000.000000,-5000.000000
1,8,Amazon,AI Engineer,120000,2026-01-18,115000.000000,5000.000000
2,1,Equifax,Data Engineer,85000,2026-01-05,91666.666667,-6666.666667
3,2,Equifax,Data Engineer,92000,2026-01-12,91666.666667,333.333333
4,3,Equifax,AI Engineer,98000,2026-01-08,91666.666667,6333.333333
5,4,Google,Data Engineer,130000,2026-01-03,133333.333333,-3333.333333
6,5,Google,Data Engineer,125000,2026-01-20,133333.333333,-8333.333333
7,6,Google,AI Engineer,145000,2026-01-15,133333.333333,11666.666667
8,9,Microsoft,Data Engineer,115000,2026-01-10,123666.666667,-8666.666667
9,10,Microsoft,AI Engineer,128000,2026-01-22,123666.666667,4333.333333
